### TP : Open Data Profiling, Data Quality Analysis, Transformations, Modeling and Medallion Architecture

##### Part 1 : Data Profiling and Data Quality Analysis

#### Imports & chargement des données

In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")

caract    = pd.read_csv(DATA_DIR / "caract-2024.csv", sep=";", low_memory=False)
lieux     = pd.read_csv(DATA_DIR / "lieux-2024.csv", sep=";", low_memory=False)
usagers   = pd.read_csv(DATA_DIR / "usagers-2024.csv", sep=";", low_memory=False)
vehicules = pd.read_csv(DATA_DIR / "vehicules-2024.csv", sep=";", low_memory=False)

datasets = {"caract": caract, "lieux": lieux, "usagers": usagers, "vehicules": vehicules}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]} lignes, {df.shape[1]} colonnes")

caract: 54402 lignes, 15 colonnes
lieux: 70248 lignes, 18 colonnes
usagers: 125187 lignes, 16 colonnes
vehicules: 92678 lignes, 11 colonnes


##### A. Dataset Structure

In [7]:
for name, df in datasets.items():
    print(f"\n=== {name} ({df.shape[0]} lignes, {df.shape[1]} colonnes) ===")
    display(df.dtypes.to_frame(name="dtype"))


=== caract (54402 lignes, 15 colonnes) ===


,dtype
Num_Acc,int64
jour,int64
mois,int64
an,int64
hrmn,object
lum,int64
dep,object
com,object
agg,int64
int,int64



=== lieux (70248 lignes, 18 colonnes) ===


,dtype
Num_Acc,int64
catr,int64
voie,object
v1,int64
v2,object
circ,int64
nbv,object
vosp,int64
prof,int64
pr,object



=== usagers (125187 lignes, 16 colonnes) ===


,dtype
Num_Acc,int64
id_usager,object
id_vehicule,object
num_veh,object
place,int64
catu,int64
grav,int64
sexe,int64
an_nais,float64
trajet,int64



=== vehicules (92678 lignes, 11 colonnes) ===


,dtype
Num_Acc,int64
id_vehicule,object
num_veh,object
senc,int64
catv,int64
obs,int64
obsm,int64
choc,int64
manv,int64
motor,int64


In [ ]:
##A.2 Semantic meaning
data_dict = {
    "caract": {
        "Num_Acc": "Identifiant unique de l'accident",
        "jour": "Jour de l'accident", "mois": "Mois", "an": "Année",
        "hrmn": "Heure et minute de l'accident",
        "lum": "Conditions de luminosité (jour, nuit, crépuscule...)",
        "dep": "Département", "com": "Commune (code INSEE)",
        "agg": "Localisation (agglomération / hors agglomération)",
        "int": "Type d'intersection", "atm": "Conditions atmosphériques",
        "col": "Type de collision", "adr": "Adresse postale",
        "lat": "Latitude GPS", "long": "Longitude GPS",
    },
    "lieux": {
        "Num_Acc": "Clé de jointure vers caract",
        "catr": "Catégorie de route", "voie": "Numéro de la route",
        "circ": "Régime de circulation", "nbv": "Nombre de voies",
        "vosp": "Voie réservée", "prof": "Profil en long (plat, pente)",
        "plan": "Tracé en plan (ligne droite, courbe)",
        "surf": "État de la surface (sèche, mouillée, verglacée)",
        "infra": "Aménagement / infrastructure", "situ": "Situation de l'accident",
        "vma": "Vitesse maximale autorisée",
    },
    "usagers": {
        "Num_Acc": "Clé de jointure vers caract", "id_vehicule": "Clé de jointure vers vehicules",
        "place": "Place occupée dans le véhicule",
        "catu": "Catégorie d'usager (conducteur/passager/piéton)",
        "grav": "Gravité (1=indemne, 2=tué, 3=hospitalisé, 4=blessé léger)",
        "sexe": "Sexe", "an_nais": "Année de naissance",
        "trajet": "Motif du déplacement",
        "secu1": "Équipement de sécurité 1", "secu2": "Équipement de sécurité 2",
        "locp": "Localisation du piéton", "actp": "Action du piéton",
    },
    "vehicules": {
        "Num_Acc": "Clé de jointure vers caract", "id_vehicule": "Clé de jointure vers usagers",
        "senc": "Sens de circulation", "catv": "Catégorie du véhicule",
        "obs": "Obstacle fixe heurté", "obsm": "Obstacle mobile heurté",
        "choc": "Point de choc initial", "manv": "Manœuvre principale",
        "motor": "Type de motorisation",
    },
}

for table, cols in data_dict.items():
    print(f"\n--- {table} ---")
    df = datasets[table]
    for col, desc in cols.items():
        marker = "OK" if col in df.columns else "absent chez toi"
        print(f"  {col:14s} [{marker:15s}] : {desc}")


--- caract ---
  Num_Acc        [OK             ] : Identifiant unique de l'accident
  jour           [OK             ] : Jour de l'accident
  mois           [OK             ] : Mois
  an             [OK             ] : Année
  hrmn           [OK             ] : Heure et minute de l'accident
  lum            [OK             ] : Conditions de luminosité (jour, nuit, crépuscule...)
  dep            [OK             ] : Département
  com            [OK             ] : Commune (code INSEE)
  agg            [OK             ] : Localisation (agglomération / hors agglomération)
  int            [OK             ] : Type d'intersection
  atm            [OK             ] : Conditions atmosphériques
  col            [OK             ] : Type de collision
  adr            [OK             ] : Adresse postale
  lat            [OK             ] : Latitude GPS
  long           [OK             ] : Longitude GPS

--- lieux ---
  Num_Acc        [OK             ] : Clé de jointure vers caract
  catr       

#### B. Missing Values and Completeness

In [9]:
##Missing percentages
missing_reports = {}
for name, df in datasets.items():
    pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
    pct = pct[pct > 0]
    missing_reports[name] = pct
    print(f"\n=== % manquant — {name} ===")
    print(pct if len(pct) > 0 else "Aucune valeur manquante")


=== % manquant — caract ===
adr    4.25
dtype: float64

=== % manquant — lieux ===
lartpc    99.95
v2        91.58
voie      18.98
dtype: float64

=== % manquant — usagers ===
an_nais    2.06
dtype: float64

=== % manquant — vehicules ===
occutc    98.98
dtype: float64


In [10]:
#Critical missingness
critical_cols = {
    "caract": ["lat", "long", "hrmn"],
    "usagers": ["grav", "an_nais", "sexe"],
    "lieux": ["vma", "surf"],
}
for table, cols in critical_cols.items():
    df = datasets[table]
    for col in cols:
        if col in df.columns:
            pct = df[col].isna().mean() * 100
            flag = " -> CRITIQUE (>5%)" if pct > 5 else ""
            print(f"{table}.{col}: {pct:.2f}% manquant{flag}")
        else:
            print(f"{table}.{col}: colonne absente")

caract.lat: 0.00% manquant
caract.long: 0.00% manquant
caract.hrmn: 0.00% manquant
usagers.grav: 0.00% manquant
usagers.an_nais: 2.06% manquant
usagers.sexe: 0.00% manquant
lieux.vma: 0.00% manquant
lieux.surf: 0.00% manquant


In [11]:
#Remediation strategies
remediation = pd.DataFrame([
    {"colonne": "lat/long", "problème": "coordonnées manquantes ou nulles",
     "action": "exclure de l'analyse spatiale ou géocoder via l'adresse (adr)"},
    {"colonne": "an_nais", "problème": "âge non calculable",
     "action": "imputer par la médiane par catégorie d'usager, ou marquer 'inconnu'"},
    {"colonne": "grav", "problème": "gravité manquante",
     "action": "exclure la ligne (variable cible, ne pas imputer)"},
    {"colonne": "vma", "problème": "vitesse max manquante",
     "action": "imputer par la médiane de la catégorie de route (catr)"},
])
display(remediation)

,colonne,problème,action
0,lat/long,coordonnées manquantes ou nulles,exclure de l'analyse spatiale ou géocoder via ...
1,an_nais,âge non calculable,"imputer par la médiane par catégorie d'usager,..."
2,grav,gravité manquante,"exclure la ligne (variable cible, ne pas imputer)"
3,vma,vitesse max manquante,imputer par la médiane de la catégorie de rout...


Les NaN bruts montrent presque aucune donnée manquante (juste adr, an_nais, et quelques colonnes rares). Mais c'est trompeur : dans ce dataset, les infos inconnues sont codées -1, pas laissées vides. En comptant aussi les -1, le vrai taux de données manquantes est bien plus élevé, notamment sur les coordonnées GPS. Il faut donc connaître le codage du dataset pour évaluer correctement sa qualité.

In [15]:
# Gestion des valeurs manquantes codées
# Exemple -1, 0 ou "" indiquent parfois une valeur manquante
sentinel_values = [-1, "-1", "", " "]

def missing_report_extended(df, name):
    true_na = df.isna().mean() * 100
    sentinel_na = df.isin(sentinel_values).mean() * 100
    total = (true_na + sentinel_na).round(2).sort_values(ascending=False)
    total = total[total > 0]
    print(f"\n=== % manquant total (NaN + codes sentinelles) — {name} ===")
    print(total if len(total) > 0 else "Aucune valeur manquante détectée")
    return total

missing_reports_ext = {name: missing_report_extended(df, name) for name, df in datasets.items()}


=== % manquant total (NaN + codes sentinelles) — caract ===
adr    4.25
col    0.01
dtype: float64

=== % manquant total (NaN + codes sentinelles) — lieux ===
lartpc    99.95
v2        91.58
v1        23.16
voie      18.98
circ       6.20
vosp       5.45
vma        5.17
infra      1.16
prof       0.07
plan       0.06
situ       0.05
surf       0.05
dtype: float64

=== % manquant total (NaN + codes sentinelles) — usagers ===
etatp      91.81
secu3      90.37
locp       49.34
secu2      42.99
trajet      2.10
an_nais     2.06
sexe        1.91
secu1       1.68
dtype: float64

=== % manquant total (NaN + codes sentinelles) — vehicules ===
occutc    98.98
motor      0.21
senc       0.07
choc       0.05
obs        0.03
obsm       0.03
manv       0.03
dtype: float64


#### C. Consistency and Validity Checks

In [17]:
# Value ranges
range_checks = {
    "caract": {"lat": (-90, 90), "long": (-180, 180)},
    "lieux": {"vma": (0, 150), "nbv": (0, 10)},
    "usagers": {"age": (0, 120)},   # calculée juste après
    "vehicules": {},  # pas de plage numérique simple ici
}

usagers["age"] = 2024 - pd.to_numeric(usagers["an_nais"], errors="coerce")

for name, checks in range_checks.items():
    df = datasets[name]
    for col, (lo, hi) in checks.items():
        if col not in df.columns:
            print(f"{name}.{col} : colonne absente")
            continue
        vals = pd.to_numeric(df[col], errors="coerce")
        invalid = df[(vals < lo) | (vals > hi) | (vals == 0)] if col in ["lat", "long"] else df[(vals < lo) | (vals > hi)]
        print(f"{name}.{col} : {len(invalid)} valeurs hors plage [{lo},{hi}] "
              f"({len(invalid)/len(df)*100:.2f}%)")

caract.lat : 0 valeurs hors plage [-90,90] (0.00%)
caract.long : 0 valeurs hors plage [-180,180] (0.00%)
lieux.vma : 3658 valeurs hors plage [0,150] (5.21%)
lieux.nbv : 4193 valeurs hors plage [0,10] (5.97%)
usagers.age : 0 valeurs hors plage [0,120] (0.00%)


In [18]:
# Categorical anomalies
categorical_checks = {
    "caract": {"lum": [1,2,3,4,5], "agg": [1,2]},
    "lieux": {"circ": [1,2,3,4,-1], "surf": list(range(1,10)) + [-1]},
    "usagers": {"grav": [1,2,3,4], "sexe": [1,2]},
    "vehicules": {"senc": [0,1,2,3,-1]},
}

for name, checks in categorical_checks.items():
    df = datasets[name]
    for col, valid_values in checks.items():
        if col not in df.columns:
            print(f"{name}.{col} : colonne absente")
            continue
        invalid = df[~df[col].isin(valid_values) & df[col].notna()]
        print(f"{name}.{col} : {df[col].nunique()} valeurs uniques, {len(invalid)} hors nomenclature")

caract.lum : 5 valeurs uniques, 0 hors nomenclature
caract.agg : 2 valeurs uniques, 0 hors nomenclature
lieux.circ : 5 valeurs uniques, 0 hors nomenclature
lieux.surf : 10 valeurs uniques, 0 hors nomenclature
usagers.grav : 4 valeurs uniques, 0 hors nomenclature
usagers.sexe : 3 valeurs uniques, 2395 hors nomenclature
vehicules.senc : 5 valeurs uniques, 0 hors nomenclature


In [20]:
# Duplicates
for name, df in datasets.items():
    dups_full = df.duplicated().sum()
    key = "Num_Acc" if "Num_Acc" in df.columns else None
    dups_key = df.duplicated(subset=key).sum() if key else "N/A"
    print(f"{name} : {dups_full} doublons stricts, {dups_key} doublons sur '{key}'")


caract : 0 doublons stricts, 0 doublons sur 'Num_Acc'
lieux : 2 doublons stricts, 15846 doublons sur 'Num_Acc'
usagers : 0 doublons stricts, 70785 doublons sur 'Num_Acc'
vehicules : 0 doublons stricts, 38276 doublons sur 'Num_Acc'


#### D. Data Quality Summary

In [25]:
#Quality report
quality_summary = pd.DataFrame([
    {"table": "caract", "lignes": len(caract), "problème_principal": "aucun majeur", "gravité": "faible"},
    {"table": "lieux", "lignes": len(lieux), "problème_principal": "vma et nbv hors plage (~5-6%), plusieurs lignes/accident (intersections, normal)", "gravité": "moyenne"},
    {"table": "usagers", "lignes": len(usagers), "problème_principal": "an_nais manquant (2,06%)", "gravité": "faible"},
    {"table": "vehicules", "lignes": len(vehicules), "problème_principal": "aucun majeur", "gravité": "faible"},
])
display(quality_summary)

,table,lignes,problème_principal,gravité
0,caract,54402,aucun majeur,faible
1,lieux,70248,"vma et nbv hors plage (~5-6%), plusieurs ligne...",moyenne
2,usagers,125187,"an_nais manquant (2,06%)",faible
3,vehicules,92678,aucun majeur,faible


Les quatre tables sont globalement propres. Les coordonnées GPS (latitude/longitude) sont toutes valides et l'âge des usagers ne présente pas de valeurs aberrantes. Le seul manque notable concerne la colonne an_nais, absente pour 2,06 % des usagers.
Le principal point d'attention se trouve dans la table lieux. La vitesse maximale autorisée (vma) est en dehors de la plage attendue pour 5,21 % des lignes, et le nombre de voies (nbv) pour 5,97 %. Il s'agit probablement de codes utilisés pour signaler une valeur inconnue ou non renseignée (par exemple -1 ou 999), plutôt que de véritables erreurs de saisie. Il est donc préférable de vérifier leur signification avant de les considérer comme des anomalies.
Concernant les doublons, la table caract n'en contient aucun, ce qui est logique puisqu'une ligne correspond à un accident. Les tables lieux, usagers et vehicules comportent en revanche plusieurs lignes avec le même Num_Acc (respectivement 15 846, 70 785 et 38 276). Pour usagers et vehicules, ce comportement est normal puisqu'un même accident peut impliquer plusieurs personnes et plusieurs véhicules. Pour lieux, la vérification montre qu'il ne s'agit pas non plus de véritables doublons : les lignes diffèrent (voie, tracé, nombre de voies), et correspondent aux accidents survenus à une intersection, où chaque voie convergente est décrite sur une ligne distincte. Aucune des trois tables ne présente donc d'anomalie de duplication réelle ; le choix à faire pour la suite est plutôt une décision de modélisation, à savoir comment ramener lieux à une seule ligne par accident pour la table de faits.

##### Part 2 :Transformations, Modeling and Medallion Architecture

##### A. Required Transformations (Silver Layer)

A.1 Standardization

In [46]:
#Standardisation
caract_silver = caract.copy()
lieux_silver = lieux.copy()
usagers_silver = usagers.copy()
vehicules_silver = vehicules.copy()

# date fusionnée en un seul champ standard
caract_silver["date"] = pd.to_datetime(
    dict(year=caract_silver["an"], month=caract_silver["mois"], day=caract_silver["jour"]),
    errors="coerce"
)
caract_silver["lat"] = pd.to_numeric(caract_silver["lat"], errors="coerce")
caract_silver["long"] = pd.to_numeric(caract_silver["long"], errors="coerce")

# types numériques garantis
lieux_silver["vma"] = pd.to_numeric(lieux_silver["vma"], errors="coerce")
lieux_silver["nbv"] = pd.to_numeric(lieux_silver["nbv"], errors="coerce")

# catégories lisibles
usagers_silver["sexe"] = usagers_silver["sexe"].map({1: "Homme", 2: "Femme"})
usagers_silver["grav"] = usagers_silver["grav"].map({1: "Indemne", 2: "Tué", 3: "Hospitalisé", 4: "Blessé léger"})

# type catégoriel
vehicules_silver["catv"] = vehicules_silver["catv"].astype("category")

A.2 Cleaning

In [47]:
# Nettoyage
n_bad_coords = caract_silver[(caract_silver["lat"] == 0) | (caract_silver["long"] == 0)].shape[0]
print(f"caract : {n_bad_coords} coordonnées invalides")

# lieux : vma/nbv hors plage (5-6% constaté) -> mis en NaN plutôt que supprimés
lieux_silver.loc[~lieux_silver["vma"].between(0, 150), "vma"] = pd.NA
lieux_silver.loc[~lieux_silver["nbv"].between(0, 10), "nbv"] = pd.NA
print(f"lieux : vma manquant = {lieux_silver['vma'].isna().mean()*100:.2f}%")
print(f"lieux : nbv manquant = {lieux_silver['nbv'].isna().mean()*100:.2f}%")

# usagers : âge calculé et bornes vérifiées (an_nais manquant pour 2,06%)
usagers_silver["age"] = 2024 - pd.to_numeric(usagers["an_nais"], errors="coerce")
usagers_silver.loc[~usagers_silver["age"].between(0, 120), "age"] = pd.NA
print(f"usagers : age manquant = {usagers_silver['age'].isna().mean()*100:.2f}%")

# vehicules : codes -1 (non renseigné) conservés tels quels
print("vehicules : codes -1 conservés comme catégorie 'non renseigné'")

caract : 0 coordonnées invalides
lieux : vma manquant = 5.21%
lieux : nbv manquant = 6.04%
usagers : age manquant = 2.06%
vehicules : codes -1 conservés comme catégorie 'non renseigné'


A.3 Enrichment

In [48]:
# Enrichissement
# caract : severity_index + time_of_day
severity_by_acc = usagers.groupby("Num_Acc")["grav"].max().rename("severity_index")
caract_silver = caract_silver.merge(severity_by_acc, on="Num_Acc", how="left")

def time_of_day(hrmn):
    try:
        h = int(str(hrmn).split(":")[0])
    except (ValueError, IndexError):
        return "inconnu"
    if 6 <= h < 12: return "matin"
    if 12 <= h < 18: return "après-midi"
    if 18 <= h < 22: return "soirée"
    return "nuit"

caract_silver["time_of_day"] = caract_silver["hrmn"].apply(time_of_day)

# catégorie de route lisible
route_labels = {1: "Autoroute", 2: "Nationale", 3: "Départementale", 4: "Voie communale", 9: "Autre"}
lieux_silver["catr_label"] = lieux_silver["catr"].map(route_labels).fillna("Autre")

# tranche d'âge dérivée
usagers_silver["age_bucket"] = pd.cut(
    usagers_silver["age"], bins=[0, 18, 25, 40, 60, 120],
    labels=["0-18", "19-25", "26-40", "41-60", "60+"]
)

# indicateur deux-roues
two_wheelers = [1, 2, 30, 31, 32, 33, 34]
vehicules_silver["is_two_wheeler"] = vehicules_silver["catv"].isin(two_wheelers)

A.4 Deduplication

In [49]:
# Déduplication
for name, df in {
    "caract": caract_silver, "lieux": lieux_silver,
    "usagers": usagers_silver, "vehicules": vehicules_silver
}.items():
    print(f"{name} : {len(df)} lignes, {df.duplicated().sum()} doublons stricts")

# Seul lieux nécessite une vraie déduplication (plusieurs lignes/accident = intersections)
lieux_silver = lieux_silver.drop_duplicates(subset="Num_Acc", keep="first")
print(f"\nlieux après dédup : {len(lieux_silver)} lignes (voie principale conservée)")
# caract, usagers, vehicules : pas de déduplication nécessaire

caract : 54402 lignes, 0 doublons stricts
lieux : 70248 lignes, 2 doublons stricts
usagers : 125187 lignes, 0 doublons stricts
vehicules : 92678 lignes, 0 doublons stricts

lieux après dédup : 54402 lignes (voie principale conservée)


A.5 Documentation

In [50]:
#Documentation des transformations
transformation_log = pd.DataFrame([
    {"table": "caract", "étape": "Standardisation", "détail": "jour/mois/an fusionnés en date"},
    {"table": "caract", "étape": "Enrichissement", "détail": "severity_index + time_of_day ajoutés"},
    {"table": "lieux", "étape": "Nettoyage", "détail": "vma/nbv hors plage -> NaN"},
    {"table": "lieux", "étape": "Enrichissement", "détail": "catr_label ajouté"},
    {"table": "lieux", "étape": "Déduplication", "détail": "1 ligne/accident gardée (voie principale)"},
    {"table": "usagers", "étape": "Nettoyage", "détail": "age calculé, bornes vérifiées"},
    {"table": "usagers", "étape": "Enrichissement", "détail": "age_bucket ajouté"},
    {"table": "vehicules", "étape": "Standardisation", "détail": "catv en type catégoriel"},
    {"table": "vehicules", "étape": "Enrichissement", "détail": "is_two_wheeler ajouté"},
])
display(transformation_log)

,table,étape,détail
0,caract,Standardisation,jour/mois/an fusionnés en date
1,caract,Enrichissement,severity_index + time_of_day ajoutés
2,lieux,Nettoyage,vma/nbv hors plage -> NaN
3,lieux,Enrichissement,catr_label ajouté
4,lieux,Déduplication,1 ligne/accident gardée (voie principale)
5,usagers,Nettoyage,"age calculé, bornes vérifiées"
6,usagers,Enrichissement,age_bucket ajouté
7,vehicules,Standardisation,catv en type catégoriel
8,vehicules,Enrichissement,is_two_wheeler ajouté


B. Modeling (Gold Layer)

In [51]:
# Fact table + dimensions
fact_accidents = caract_silver[[
    "Num_Acc", "date", "time_of_day", "lat", "long", "dep", "com",
    "lum", "atm", "col", "severity_index"
]].merge(
    lieux_silver[["Num_Acc", "catr", "catr_label", "vma", "surf"]],
    on="Num_Acc", how="left"
)

dim_location = lieux_silver[["Num_Acc", "catr", "catr_label", "vma", "nbv", "surf", "prof", "plan"]].copy()
dim_vehicle = vehicules_silver[["Num_Acc", "id_vehicule", "catv", "is_two_wheeler", "obs", "obsm", "choc"]].copy()
dim_user = usagers_silver[["Num_Acc", "id_vehicule", "catu", "grav", "sexe", "age", "age_bucket"]].copy()

dim_date = fact_accidents[["date"]].drop_duplicates().dropna()
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["weekday"] = dim_date["date"].dt.day_name()

print("fact_accidents :", fact_accidents.shape)
print("dim_location :", dim_location.shape)
print("dim_vehicle :", dim_vehicle.shape)
print("dim_user :", dim_user.shape)
print("dim_date :", dim_date.shape)

fact_accidents : (54402, 15)
dim_location : (54402, 8)
dim_vehicle : (92678, 7)
dim_user : (125187, 7)
dim_date : (366, 4)


In [52]:
#Sauvegarde des tables Gold (pour Streamlit) dashboard 
fact_accidents.to_csv("fact_accidents.csv", index=False)
dim_location.to_csv("dim_location.csv", index=False)
dim_vehicle.to_csv("dim_vehicle.csv", index=False)
dim_user.to_csv("dim_user.csv", index=False)
dim_date.to_csv("dim_date.csv", index=False)

print("Tables gold sauvegardées")

Tables gold sauvegardées


C. Medallion Architecture Diagram

```mermaid
flowchart LR
    subgraph Bronze [Bronze fichiers bruts]
        A1[caract-2024.csv]
        A2[lieux-2024.csv]
        A3[usagers-2024.csv]
        A4[vehicules-2024.csv]
    end

    subgraph Silver [Silver nettoyé, standardisé, enrichi]
        B1[caract_silver]
        B2[lieux_silver]
        B3[usagers_silver]
        B4[vehicules_silver]
    end

    subgraph Gold [Gold 
    modèle analytique]
        C1[fact_accidents]
        C2[dim_location]
        C3[dim_vehicle]
        C4[dim_user]
        C5[dim_date]
    end

    D[Dashboard Streamlit]

    A1 --> B1 --> C1
    A2 --> B2 --> C2
    A3 --> B3 --> C4
    A4 --> B4 --> C3
    C1 --> D
    C2 --> D
    C3 --> D
    C4 --> D
    C5 --> D
```